# 🏙️ 三明市城市扩张遥感分析（2000-2025）\n\n## 基于 Landsat 影像 + Google Earth Engine 的建成区提取与分析\n\n本 Notebook 带你逐步完成：\n1. GEE 初始化 & 研究区定义\n2. Landsat 影像获取 & 预处理\n3. NDBI 计算 & 建成区提取\n4. 面积统计 & 时空分析\n5. 可视化输出（地图 + 图表 + 动图）\n\n**首次使用请先**：\n- 注册 Google Earth Engine（免费）：https://earthengine.google.com/signup/\n- 终端运行认证：`earthengine authenticate`

---\n## 📦 第 1 步：导入依赖库

In [ ]:
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import os
import warnings
from PIL import Image

warnings.filterwarnings('ignore')

# 设置中文字体
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-whitegrid')

print('[✓] 依赖库导入成功')

---\n## 🌍 第 2 步：初始化 Google Earth Engine

In [ ]:
try:
    ee.Initialize()
    print('[✓] GEE 初始化成功')
except ee.EEException:
    print('[!] 首次使用需要认证')
    print('[→] 请在终端运行: earthengine authenticate')
    print('[→] 或者取消注释下一行，在 Notebook 中直接认证')
    # ee.Authenticate()  # 取消注释这行来在 Notebook 中认证
    # ee.Initialize()

---\n## 📍 第 3 步：配置参数 & 定义研究区\n\n**💡 想分析其他城市？改下面的经纬度就行！**

In [ ]:
# ==================== 配置参数（改这里！）====================
CONFIG = {
    'city_name': '三明市',
    'lon': 117.64,          # 经度
    'lat': 26.26,           # 纬度
    'buffer': 15000,        # 缓冲区半径（米）
    'years': [2000, 2005, 2010, 2015, 2020, 2025],
    'ndbi_threshold': -0.05,  # NDBI 建成区阈值
}

# 定义研究区域
ROI = ee.Geometry.Point([CONFIG['lon'], CONFIG['lat']]).buffer(CONFIG['buffer'])

# 可视化研究区域
Map = geemap.Map()
Map.centerObject(ROI, 11)
Map.addLayer(ROI, {'color': 'red'}, '研究区域')
Map

---\n## 🛰️ 第 4 步：获取 Landsat 影像 & 计算 NDBI\n\n核心公式：**NDBI = (SWIR1 - NIR) / (SWIR1 + NIR)**

In [ ]:
def mask_cloud_landsat5(image):
    """Landsat 5 云掩膜"""
    qa = image.select('QA_PIXEL') if 'QA_PIXEL' in image.bandNames().getInfo() else image.select('QA60')
    cloud_mask = qa.bitwiseAnd(1 << 5).eq(0)
    return image.updateMask(cloud_mask)


def mask_cloud_landsat8(image):
    """Landsat 8 云掩膜"""
    qa = image.select('QA_PIXEL')
    cloud_mask = qa.bitwiseAnd(1 << 3).eq(0)      # 云
    shadow_mask = qa.bitwiseAnd(1 << 4).eq(0)     # 云阴影
    return image.updateMask(cloud_mask.And(shadow_mask))


def get_landsat_image(year, roi):
    """获取指定年份的 Landsat 影像"""
    start_date = f'{year}-01-01'
    end_date = f'{year}-12-31'
    
    if year <= 2012:
        collection = (ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
                      .filterBounds(roi)
                      .filterDate(start_date, end_date)
                      .filter(ee.Filter.lt('CLOUD_COVER', 30))
                      .map(mask_cloud_landsat5)
                      .median())
        image = collection.select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'])
        sensor = 'L5'
    else:
        collection = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
                      .filterBounds(roi)
                      .filterDate(start_date, end_date)
                      .filter(ee.Filter.lt('CLOUD_COVER', 30))
                      .map(mask_cloud_landsat8)
                      .median())
        image = collection.select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'])
        sensor = 'L8'
    
    return image.clip(roi), sensor


def calc_ndbi(image):
    """计算 NDBI（归一化建筑指数）"""
    nir = image.select([3])       # 近红外
    swir1 = image.select([4])     # 中红外1
    ndbi = swir1.subtract(nir).divide(swir1.add(nir)).rename('NDBI')
    return image.addBands(ndbi)


def extract_built_up(image, threshold=-0.05):
    """用 NDBI 阈值提取建成区"""
    ndbi = image.select('NDBI')
    return ndbi.gt(threshold).rename('built_up')


print('[✓] 函数定义完成')

---\n## 🔬 第 5 步：逐年份分析\n\n循环处理每一年：获取影像 → 计算 NDBI → 提取建成区 → 计算面积

In [ ]:
# 创建输出目录
os.makedirs('../outputs/maps', exist_ok=True)
os.makedirs('../outputs/charts', exist_ok=True)
os.makedirs('../outputs/stats', exist_ok=True)

stats_list = []
truecolor_images = {}
builtup_images = {}

for year in CONFIG['years']:
    print(f'\n{'='*50}')
    print(f'[→] 处理 {year} 年...')
    
    # 获取影像
    image, sensor = get_landsat_image(year, ROI)
    print(f'    传感器: Landsat {sensor}')
    
    # 计算 NDBI
    image_with_ndbi = calc_ndbi(image)
    
    # 提取建成区
    built_up = extract_built_up(image_with_ndbi, CONFIG['ndbi_threshold'])
    
    # 计算面积
    pixel_area = built_up.multiply(ee.Image.pixelArea())
    stats = pixel_area.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=ROI,
        scale=30,
        maxPixels=1e9
    )
    area_km2 = round(ee.Number(stats.get('built_up')).divide(1e6).getInfo(), 2)
    print(f'    建成区面积: {area_km2} km²')
    
    stats_list.append({'年份': year, '建成区面积(km²)': area_km2})
    truecolor_images[year] = image
    builtup_images[year] = built_up

print(f'\n[✓] 全部年份处理完成！')

---\n## 🗺️ 第 6 步：查看某一年份的地图\n\n调整 `year` 变量查看不同年份

In [ ]:
# 选择要查看的年份
year = 2025  # ← 改这里！可选: 2000, 2005, 2010, 2015, 2020, 2025

# 创建设置地图
display_map = geemap.Map()
display_map.centerObject(ROI, 11)

# 真彩色影像
display_map.addLayer(
    truecolor_images[year],
    {'bands': [2, 1, 0], 'min': 8000, 'max': 18000, 'gamma': 1.4},
    f'{year}年 真彩色影像'
)

# 建成区叠加（红色）
display_map.addLayer(
    builtup_images[year].updateMask(builtup_images[year]),
    {'palette': ['red'], 'opacity': 0.5},
    f'{year}年 建成区'
)

display_map

---\n## 📊 第 7 步：统计分析 & 图表

In [ ]:
# 创建统计表
df = pd.DataFrame(stats_list)
df['增长率(%)'] = df['建成区面积(km²)'].pct_change() * 100
df['增长率(%)'] = df['增长率(%)'].round(1)

# 保存 CSV
csv_path = '../outputs/stats/面积统计表.csv'
df.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f'[✓] 统计表已保存: {csv_path}')
print()
print(df.to_string(index=False))

In [ ]:
# 建成区面积年际变化柱状图
fig, ax = plt.subplots(figsize=(10, 6))

years = df['年份'].tolist()
areas = df['建成区面积(km²)'].tolist()
colors = plt.cm.YlOrRd(np.linspace(0.4, 0.9, len(years)))

bars = ax.bar(years, areas, color=colors, edgecolor='white', linewidth=1.2, width=2.5)

for bar, area in zip(bars, areas):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(areas)*0.01,
            f'{area:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xlabel('年份', fontsize=13)
ax.set_ylabel('建成区面积 (km²)', fontsize=13)
ax.set_title(f'{CONFIG["city_name"]}建成区面积年际变化（2000-2025）', fontsize=15, fontweight='bold')

plt.tight_layout()
chart_path = '../outputs/charts/建成区面积年际变化.png'
plt.savefig(chart_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'[✓] 图表已保存: {chart_path}')

In [ ]:
# 各阶段扩张速率折线图
periods = []
rates = []
for i in range(1, len(years)):
    period = f'{years[i-1]}-{years[i]}'
    rate = (areas[i] - areas[i-1]) / (years[i] - years[i-1])
    periods.append(period)
    rates.append(rate)

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(periods, rates, 'o-', color='#E74C3C', linewidth=2.5, markersize=10,
         markerfacecolor='white', markeredgewidth=2)
ax1.fill_between(range(len(periods)), rates, alpha=0.15, color='#E74C3C')
ax1.set_ylabel('年均扩张速率 (km²/年)', fontsize=13, color='#E74C3C')
ax1.tick_params(axis='y', labelcolor='#E74C3C')

for i, rate in enumerate(rates):
    ax1.annotate(f'{rate:.2f}', (i, rate), textcoords='offset points',
                 xytext=(0, 15), ha='center', fontsize=11, fontweight='bold')

ax1.set_xlabel('时间段', fontsize=13)
ax1.set_title(f'{CONFIG["city_name"]}各阶段城市扩张速率', fontsize=15, fontweight='bold')

plt.tight_layout()
rate_path = '../outputs/charts/各阶段扩张速率.png'
plt.savefig(rate_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'[✓] 图表已保存: {rate_path}')

---\n## 🎬 第 8 步（可选）：对比多年份变化

In [ ]:
# 左右对比 2000 vs 2025
map_compare = geemap.Map()

# 左侧：2000 年 + 建成区
left_layer = geemap.ee_tile_layer(
    truecolor_images[2000],
    {'bands': [2, 1, 0], 'min': 8000, 'max': 18000, 'gamma': 1.4},
    '2000年'
)
map_compare.split_map(
    left_layer,
    geemap.ee_tile_layer(truecolor_images[2025],
                         {'bands': [2, 1, 0], 'min': 8000, 'max': 18000, 'gamma': 1.4},
                         '2025年')
)
map_compare.centerObject(ROI, 11)
map_compare

---\n## ✅ 分析总结\n\n运行完所有单元格后：\n- 📁 `outputs/maps/` — 交互地图（HTML）\n- 📁 `outputs/charts/` — 统计图表（PNG）\n- 📁 `outputs/stats/` — 面积数据（CSV）\n\n**下一步**：\n- 修改 `CONFIG` 里的经纬度，分析其他城市\n- 调整 `ndbi_threshold` 优化提取精度\n- 运行 `src/urban_analysis.py` 一键生成全部输出